# Frequent Itemset Mining

A comparison of three market basket analysis algorithms — **Brute Force**, **Apriori**, and **FP-Growth** — across five retail transaction datasets (Amazon, Best Buy, Kroger, Target, Walmart).

---

| Algorithm | Strategy |
|-----------|----------|
| Brute Force | Exhaustive combinatorial search — all itemset sizes |
| Apriori | Level-wise candidate generation and downward-closure pruning |
| FP-Growth | FP-tree compression — no candidate generation phase |

Key metrics: execution time, frequent itemsets found, association rules generated.

In [ ]:
import pandas as pd
import itertools
import os
import time
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder

## Data Loading

In [ ]:
class DataManager:
    """Loads and verifies the five retail transaction datasets."""

    @staticmethod
    def get_available_datasets():
        datasets = []
        for name in ['amazon', 'bestbuy', 'walmart', 'target', 'kroger']:
            if os.path.exists(f"{name}.csv"):
                datasets.append(name)
        return sorted(datasets)

    @staticmethod
    def load_dataset(dataset_name):
        filename = f"{dataset_name}.csv"
        if not os.path.exists(filename):
            raise FileNotFoundError(f"Dataset {filename} not found")
        df = pd.read_csv(filename)
        return [row['items'].split(',') for _, row in df.iterrows()]


# Verify all datasets are present
available = DataManager.get_available_datasets()
print("Available datasets:", available)
assert len(available) == 5, "Expected 5 datasets — check working directory."

## Brute Force Algorithm

In [ ]:
class BruteForceAlgorithm:
    """Exhaustive frequent itemset mining — no pruning."""

    def __init__(self, transactions):
        self.transactions = transactions
        self.num_transactions = len(transactions)
        self.frequent_itemsets = {}
        self.association_rules = []

    def get_unique_items(self):
        unique = set()
        for t in self.transactions:
            unique.update(t)
        return sorted(unique)

    def calculate_support(self, itemset):
        count = sum(1 for t in self.transactions if set(itemset).issubset(set(t)))
        return count / self.num_transactions

    def find_frequent_itemsets(self, min_support):
        unique_items = self.get_unique_items()
        self.frequent_itemsets = {}
        k = 1
        while k <= len(unique_items):
            frequent_k = [
                {'itemset': item, 'support': self.calculate_support(item)}
                for item in itertools.combinations(unique_items, k)
                if self.calculate_support(item) >= min_support
            ]
            if not frequent_k:
                break
            self.frequent_itemsets[k] = frequent_k
            k += 1
        return self.frequent_itemsets

    def generate_association_rules(self, min_confidence):
        self.association_rules = []
        for k in range(2, len(self.frequent_itemsets) + 1):
            for info in self.frequent_itemsets[k]:
                itemset, itemset_support = info['itemset'], info['support']
                for i in range(1, len(itemset)):
                    for ant in itertools.combinations(itemset, i):
                        ant_support = self.calculate_support(tuple(ant))
                        if ant_support > 0:
                            conf = itemset_support / ant_support
                            if conf >= min_confidence:
                                self.association_rules.append({
                                    'antecedent': tuple(ant),
                                    'consequent': tuple(set(itemset) - set(ant)),
                                    'support': itemset_support,
                                    'confidence': conf
                                })
        return self.association_rules

## Apriori & FP-Growth (mlxtend)

In [ ]:
class LibraryAlgorithms:
    """Apriori and FP-Growth via mlxtend."""

    @staticmethod
    def _encode(transactions):
        te = TransactionEncoder()
        return pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)

    @staticmethod
    def run_apriori(transactions, min_support, min_confidence):
        df = LibraryAlgorithms._encode(transactions)
        itemsets = apriori(df, min_support=min_support, use_colnames=True)
        if len(itemsets) == 0:
            return itemsets, pd.DataFrame()
        rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)
        return itemsets, rules

    @staticmethod
    def run_fpgrowth(transactions, min_support, min_confidence):
        df = LibraryAlgorithms._encode(transactions)
        itemsets = fpgrowth(df, min_support=min_support, use_colnames=True)
        if len(itemsets) == 0:
            return itemsets, pd.DataFrame()
        rules = association_rules(itemsets, metric='confidence', min_threshold=min_confidence)
        return itemsets, rules

## Result Display Utilities

In [ ]:
class ResultDisplayer:
    """Formats and prints algorithm results."""

    @staticmethod
    def display_algorithm_results(name, elapsed, itemsets_count, rules_count,
                                   sample_itemsets=None, sample_rules=None):
        print(f"\n{'=' * 50}")
        print(f"  {name}")
        print(f"{'=' * 50}")
        print(f"  Execution Time   : {elapsed:.4f} s")
        print(f"  Frequent Sets    : {itemsets_count}")
        print(f"  Association Rules: {rules_count}")
        if itemsets_count == 0:
            print("  No frequent itemsets — try a lower support threshold.")
            return
        if sample_itemsets:
            print("\n  Top Frequent Itemsets:")
            for i, info in enumerate(sample_itemsets[:5], 1):
                if isinstance(info, dict) and 'itemset' in info:
                    items_str = ', '.join(info['itemset'])
                    sup = info['support']
                else:
                    items_str = ', '.join(list(info['itemsets']))
                    sup = info['support']
                print(f"    {i}. {{{items_str}}}  support={sup:.3f}")
        if rules_count > 0 and sample_rules:
            print("\n  Top Association Rules:")
            for i, rule in enumerate(sample_rules[:5], 1):
                if 'antecedent' in rule:
                    ant = ', '.join(rule['antecedent'])
                    con = ', '.join(rule['consequent'])
                    sup, conf = rule['support'], rule['confidence']
                else:
                    ant = ', '.join(list(rule['antecedents']))
                    con = ', '.join(list(rule['consequents']))
                    sup, conf = rule['support'], rule['confidence']
                print(f"    {i}. {{{ant}}} => {{{con}}}  sup={sup:.3f}  conf={conf:.3f}")
        elif rules_count == 0:
            print("  No rules — try a lower confidence threshold.")

    @staticmethod
    def display_comparison(results):
        print("\n" + "=" * 60)
        print("  PERFORMANCE COMPARISON")
        print("=" * 60)
        print(f"  {'Algorithm':<15} {'Time (s)':<12} {'Itemsets':<12} {'Rules'}")
        print(f"  {'-' * 50}")
        for algo, r in results.items():
            print(f"  {algo:<15} {r['time']:<12.4f} {r['itemsets']:<12} {r['rules']}")

## Example Run

Runs all three algorithms on the **Amazon** dataset with `min_support=0.2` and `min_confidence=0.5`.

> Change the three variables below to explore different datasets or thresholds.

In [ ]:
# Configuration
DATASET        = 'amazon'   # amazon | bestbuy | kroger | target | walmart
MIN_SUPPORT    = 0.2
MIN_CONFIDENCE = 0.5

transactions = DataManager.load_dataset(DATASET)
print(f"Dataset: {DATASET}  |  Transactions: {len(transactions)}")
print(f"Min support: {MIN_SUPPORT}  |  Min confidence: {MIN_CONFIDENCE}")

comparison = {}

# Brute Force
t0 = time.time()
bf = BruteForceAlgorithm(transactions)
bf_sets = bf.find_frequent_itemsets(MIN_SUPPORT)
bf_rules = bf.generate_association_rules(MIN_CONFIDENCE)
bf_time = time.time() - t0
bf_all = [info for k in bf_sets for info in bf_sets[k]]
ResultDisplayer.display_algorithm_results(
    'BRUTE FORCE', bf_time, len(bf_all), len(bf_rules), bf_all, bf_rules
)
comparison['Brute Force'] = {'time': bf_time, 'itemsets': len(bf_all), 'rules': len(bf_rules)}

# Apriori
t0 = time.time()
ap_sets, ap_rules = LibraryAlgorithms.run_apriori(transactions, MIN_SUPPORT, MIN_CONFIDENCE)
ap_time = time.time() - t0
ResultDisplayer.display_algorithm_results(
    'APRIORI', ap_time, len(ap_sets), len(ap_rules),
    ap_sets.to_dict('records') if len(ap_sets) > 0 else [],
    ap_rules.to_dict('records') if len(ap_rules) > 0 else []
)
comparison['Apriori'] = {'time': ap_time, 'itemsets': len(ap_sets), 'rules': len(ap_rules)}

# FP-Growth
t0 = time.time()
fp_sets, fp_rules = LibraryAlgorithms.run_fpgrowth(transactions, MIN_SUPPORT, MIN_CONFIDENCE)
fp_time = time.time() - t0
ResultDisplayer.display_algorithm_results(
    'FP-GROWTH', fp_time, len(fp_sets), len(fp_rules),
    fp_sets.to_dict('records') if len(fp_sets) > 0 else [],
    fp_rules.to_dict('records') if len(fp_rules) > 0 else []
)
comparison['FP-Growth'] = {'time': fp_time, 'itemsets': len(fp_sets), 'rules': len(fp_rules)}

ResultDisplayer.display_comparison(comparison)

## Interactive CLI

For a fully interactive terminal experience:

```bash
python frequent_itemset_mining.py
```

This prompts you to select a dataset and enter support/confidence thresholds at runtime.